## 📑 Core Legal Actions
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/FYBOH)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col, regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ"S3PREFIX" 

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

unionRefDate =  spark.sql(f'''
                              SELECT min(BusinessDate) Ref
                              FROM `{env_prefix}-cdh-delivery`.repoint.core_mastertable m
						WHERE BusinessDate <= {run_date!r}
						AND BusinessDate > last_day(add_months({run_date!r},-2))
                              ''').collect()[0]['Ref']
unionRefDate = unionRefDate.strftime('%Y-%m-%d')

###### Schema of the table

In [0]:
table_name = "Core_LegalActions"

schema = (
    "BusinessDate date,"
    "CDI string,"
    "PartyTaxRegistrationNumber string,"
    "ParentPartyId string,"
    "AgreementKey string,"
    "ExposureKey string,"
    "ArrearsStatusENG string,"
    "AssetSegment string,"
    "Balance decimal(19, 2),"
    "OnBalanceAmount decimal(19, 2),"
    "OffBalanceAmount decimal(19, 2),"
    "AgencyCode string,"
    "AssignmentDate_dp date,"
    "AssignmentDate_agog date,"
    "AssignmentDate_ktsx date,"
    "AssignmentDate_pros date,"
    "CurrentAgency_dp string,"
    "CurrentAgency_agog string,"
    "CurrentAgency_ktsx string,"
    "CurrentAgency_pros string,"
    "`ΚΑΤΑ_ΔΠ` date,"
    "`ΕΚΔ_ΔΠ` date,"
    "`ΕΠΙ_ΔΠ` date,"
    "`ΚΑΤΑΣΧ` date,"
    "`ΚΑΤΑ_ΑΓΩΓ` date,"
    "`ΕΚΔ_ΑΓΩΓ` date,"
    "`ΚΟΙΝ_ΑΓΩΓ` date,"
    "`ΠΡΟΣΗΜ` date,"
    "`AG_ΚΑΤΑ_ΔΠ` string,"
    "`AG_ΕΚΔ_ΔΠ` string,"
    "`AG_ΕΠΙ_ΔΠ` string,"
    "`AG_ΚΑΤΑΣΧ` string,"
    "`PrimaryContract_ktsx` int,"
    "`ΚΑΤΑ_ΔΠ_Time` int,"
    "`ΕΚΔ_ΔΠ_Time` int,"
    "`ΕΠΙ_ΔΠ_Time` int,"
    "`Ekkr_ΔΠ_Time` int,"
    "`ΚΑΤΑ_ΑΓΩΓ_Time` int,"
    "`ΕΚΔ_ΑΓΩΓ_Time` int,"
    "`ΚΟΙΝ_ΑΓΩΓ_Time` int,"
    "`Ekkr_ΠΑΓ_Time` int,"
    "`ΚΑΤΑΣΧ_Time` int,"
    "`Ekkr_KΣΧ_Time` int,"
    "`ΠΡΟΣ_Time` int,"
    "`Ekkr_ΠΡΟΣ_Time` int,"
    "PopulationType string,"
    "InsertTimeStamp timestamp"																		
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"

delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date
#### type: ignore
)


##### Optimizer function 

In [0]:
%python
# Helper function to create or replace table
def create_or_replace_table(table_name, df, partition_col):
    df.write.format("delta").mode("overwrite").partitionBy(partition_col).saveAsTable(table_name)
def create_or_replace_table_no_partition(table_name, df):
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
def repair_and_analyze_table(table_name):
    spark.sql(f"REPAIR TABLE {table_name} SYNC METADATA")
    spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS")    


##### MasterTable

In [0]:
mastertable = spark.sql(
f'''
SELECT 	acc.BusinessDate, acc.CDI, acc.PartyTaxRegistrationNumber, acc.AgencyCode, acc.AgreementKey, acc.ExposureKey, acc.AssetSegment,  acc.AccountStatus, acc.OnBalanceAmount, acc.OffBalanceAmount, acc.Balance, IFNULL(acc.PartyTaxRegistrationNumber,acc.CDI) ParentPartyId, 
CASE WHEN acc.MenuCode IS NOT NULL THEN 1 ELSE 0 END CMSFlag, 
CASE WHEN acc.spv = 'GEMINI CORE SECURITISATION DAC' THEN 1 ELSE 0 END GeMINiAccFlag
FROM `{env_prefix}-cdh-delivery`.repoint.core_mastertable acc
WHERE acc.BusinessDate = {unionRefDate!r}

'''
)
mastertable.createOrReplaceTempView("mastertable")

##### LegalActivitiesStatic

##### Recursive Schema

In [0]:
from pyspark.sql.functions import col
# Recursive function to perform the UNION ALL operation
def recursive_union(name, depth, mode):
    if mode == 'Mods':
        if depth == 1:
            return spark.sql(f"SELECT DISTINCT NewExposureKey, OldExposureKey FROM {name}")
        else:
            # Perform the recursive call and alias the resulting DataFrame
            recursive_df = recursive_union(name, depth - 1, mode).alias("m")
        
            # Join the recursive DataFrame with the original DataFrame
            # You need to register the recursive DataFrame as a temporary view to use it in the SQL query
            recursive_df.createOrReplaceTempView("m")
        
            return spark.sql(f"""
                SELECT DISTINCT   NewExposureKey, OldExposureKey FROM {name}
                UNION ALL
                SELECT  m.NewExposureKey, a.OldExposureKey
                FROM {name} a
            JOIN m ON m.OldExposureKey = a.NewExposureKey AND m.NewExposureKey <> a.OldExposureKey

            """)
    elif mode == 'TimePeriods':
        if depth == 1:
            return spark.sql(f" SELECT AgreementKey, LegalActivity, StartDate, EndDate, rn, 1 as GroupId FROM {name}")
        else:
            recursive_df = recursive_union(name, depth - 1, mode).alias("p1")
        
            recursive_df.createOrReplaceTempView("p1")
        
            return spark.sql(f'''
                    SELECT  AgreementKey,  LegalActivity,  StartDate,  EndDate,  rn, 1 as GroupId FROM {name} where rn = 1
                    UNION ALL 
                    SELECT  p2.AgreementKey,
                        p1.LegalActivity,
                        CASE
                        WHEN (p1.startDate between p2.startDate AND p2.endDate) THEN p2.startDate
                        WHEN (p2.startDate between p1.startDate AND p1.endDate) THEN p1.startDate
                        WHEN (p1.startDate < p2.startDate AND p1.endDate > p2.endDate) THEN p1.startDate
                        WHEN (p1.startDate > p2.startDate AND p1.endDate < p2.endDate) THEN p2.startDate
                        ELSE p2.startDate
                        END as startDate,
                        
                        CASE
                        WHEN (p1.endDate between p2.startDate AND p2.endDate) THEN p2.endDate
                        WHEN (p2.endDate between p1.startDate AND p1.endDate) THEN p1.endDate
                        WHEN (p1.startDate < p2.startDate AND p1.endDate > p2.endDate) THEN p1.endDate
                        WHEN (p1.startDate > p2.startDate AND p1.endDate < p2.endDate) THEN p2.endDate
                        ELSE p2.endDate
                        END as endDate,
                        
                        p2.rn,
                        CASE WHEN
                        (p1.startDate between p2.startDate AND p2.endDate) or
                        (p1.endDate between p2.startDate AND p2.endDate) or
                        (p1.startDate < p2.startDate AND p1.endDate > p2.endDate) or
                        (p1.startDate > p2.startDate AND p1.endDate < p2.endDate)
                        THEN
                        p1.GroupId
                        ELSE
                        (p1.GroupId+1)
                        END as GroupId
                        FROM p1 
                        inner JOIN {name} p2
                        on p1.LegalActivity = p2.LegalActivity AND
                        p1.AgreementKey = p2.AgreementKey AND
                        (p1.rn+1) = p2.rn
                     '''
            )


##### CaseCode

In [0]:
CaseCode = spark.sql(
f'''
SELECT CaseCodeKey, Agency, ExposureKey FROM (
    SELECT CaseCodeKey, Agency, ExposureKey,  rank() over (partition by ExposureKey order by BusinessDate desc) as Rank
    FROM (
        SELECT  CaseCodeKey, Agency, SUBSTRING_INDEX(AccountNumber, '-', 1) as ExposureKey, BusinessDate
        FROM  `{env_prefix}-cdh`.primary.casecode
        WHERE 1=1 
        AND BusinessDate <= {run_date!r}
        and PortfolioName = 'Core Retail'
    )
) WHERE Rank = 1
'''
)
CaseCode.createOrReplaceTempView("CaseCode")

##### exposure

In [0]:
exposure  = spark.sql(
f'''
SELECT  
    A.BusinessDate
    , A.PortfolioName
    , A.ExposureKey
    , A.AssetSegment
    , A.AgreementKey
    , A.CDI
    , A.IsActiveFlag
    , A.CaseCode
FROM `{env_prefix}-cdh`.primary.exposure A
WHERE A.IsActiveFlag = true
AND A.PortfolioName = 'Core Retail'
AND A.BusinessDate = {run_date!r}
AND PartyRelationshipType = 'Primary Owner'
'''
)
exposure.createOrReplaceTempView("exposure")

##### Agent

In [0]:
agent = spark.sql(
f'''
SELECT e.ExposureKey, e.AgreementKey, e.CDI, c.Agency 
FROM exposure e 
LEFT JOIN CaseCode c 
ON e.CaseCode  = c.CaseCodeKey 
and  e.exposureKey = c.exposureKey
'''
)
agent.createOrReplaceTempView("Agent")

##### ExtCompany

In [0]:
extcompany  = spark.sql(
f'''
select distinct EmployeePartyKey as CreatedBy, EmployeeFullName as ExternalCompany    
from `{env_prefix}-cdh`.primary.firmstructure fs
WHERE 1=1 
AND EmployeeCategory = "External Company"
and SourceSystem = 'qca'
and fs.BusinessDate = {run_date!r}
'''
)  
extcompany.createOrReplaceTempView("extcompany")

##### Nomikes

In [0]:
Nomikes = spark.sql(
f'''
SELECT * , rank() over(partition by a.ExposureKey, LegalChapter order by a.ActivityDate asc, ExternalCompany asc, ActivityTypeKey desc) Rank 
FROM (
SELECT distinct a.CDI, a.PartyTaxRegistrationNumber, a.ParentPartyId, a.AgreementKey, a.ExposureKey, a.AssetSegment, a.CMSFlag, a.ArrearsStatusENG, a.Balance, cast(a.ActivityDate as date) ActivityDate, a.ActivityTypeKey, a.ActivityName, a.LegalActivityMapping, a.LegalChapter, a.ExternalCompany as ExternalCompany  /* la.ExternalCompany */
FROM `{env_prefix}-cdh-delivery`.repoint.int_corelegalaction  a
WHERE a.ActivityDate <= {run_date!r} AND a.ActivityDate > '2018-10-31' and a.LegalActivityCategory = 'Νομικές Ενέργειες'
) a
'''
)
Nomikes.createOrReplaceTempView("Nomikes")


##### NomikesAnagContract 

In [0]:
NomikesAnagContract = spark.sql(
f'''
SELECT * 
FROM (
SELECT distinct CDI, PartyTaxRegistrationNumber, ParentPartyId, AgreementKey, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
,rank() over(partition by AgreementKey, LegalChapter order by ActivityDate asc, ExternalCompany asc, ActivityTypeKey desc) Rank
FROM Nomikes
WHERE Rank = 1
AND CMSFlag = 0) n
WHERE n.Rank = 1
'''
)
NomikesAnagContract.createOrReplaceTempView("NomikesAnagContract")

##### NomikesAnagContractCMS

In [0]:
NomikesAnagContractCMS = spark.sql(
f'''
		SELECT * 
		FROM (
		SELECT distinct CDI, PartyTaxRegistrationNumber, ParentPartyId, AgreementKey, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
		,rank() over(partition by AgreementKey, LegalChapter order by ActivityDate asc, ExternalCompany asc, ActivityTypeKey desc) Rank
		FROM Nomikes
		WHERE Rank = 1
		AND CMSFlag = 1) n
		WHERE n.Rank = 1
'''
)
NomikesAnagContractCMS.createOrReplaceTempView("NomikesAnagContractCMS")

##### NomikesAnagAFM

In [0]:
NomikesAnagAFM = spark.sql(
f'''
		SELECT * 
		FROM (
		SELECT distinct ParentPartyId, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
		,rank() over(partition by ParentPArtyId, LegalChapter order by ActivityDate asc, ExternalCompany asc, ActivityTypeKey desc) Rank
		FROM Nomikes
		WHERE Rank = 1
		AND LegalActivityMapping = 'Κατάσχεση') n
		WHERE n.Rank = 1
  '''
)
NomikesAnagAFM.createOrReplaceTempView("NomikesAnagAFM")

##### NomikesAnag

In [0]:
NomikesAnag = spark.sql(
f'''
		SELECT * 
		FROM (
		SELECT ne.CDI, ne.PartyTaxRegistrationNumber, ne.ParentPartyId, ne.AgreementKey, acc.ExposureKey, acc.CMSFlag, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
		FROM NomikesAnagContract ne
		JOIN mastertable acc on acc.AgreementKey = ne.AgreementKey
		WHERE LegalActivityMapping <> 'Κατάσχεση'
		UNION ALL
		SELECT acc.CDI, acc.PartyTaxRegistrationNumber, acc.ParentPartyId, acc.AgreementKey, acc.ExposureKey, acc.CMSFlag, ne.ActivityDate, ne.ActivityTypeKey, ne.ActivityName, ne.LegalActivityMapping, ne.LegalChapter, ne.ExternalCompany
		FROM NomikesAnagAFM ne
		JOIN mastertable acc on acc.ParentPartyId = ne.ParentPartyId
		UNION ALL
		SELECT ne.CDI, ne.PartyTaxRegistrationNumber, ne.ParentPartyId, ne.AgreementKey, ne.ExposureKey, CMSFlag, ne.ActivityDate, ne.ActivityTypeKey, ne.ActivityName, ne.LegalActivityMapping, ne.LegalChapter, ne.ExternalCompany
		FROM Nomikes ne
		WHERE CMSFlag = 1
		AND LegalActivityMapping <> 'Κατάσχεση') a
  '''
)
NomikesAnag.createOrReplaceTempView("NomikesAnag")

#####PrimaryContractKTSX

In [0]:
PrimaryContractKTSX = spark.sql(
f'''
		SELECT distinct ne.AgreementKey, CASE WHEN c.AgreementKey IS NOT NULL THEN 1 ELSE 0 end as PrimaryContract_ktsx
		FROM NomikesAnagContract ne
		LEFT JOIN NomikesAnagContract c on c.AgreementKey = ne.AgreementKey AND c.ActivityDate = ne.ActivityDate AND c.ActivityTypeKey = ne.ActivityTypeKey AND ne.LegalActivityMapping = c.LegalActivityMapping
		WHERE ne.LegalActivityMapping = 'Κατάσχεση' 
		UNION ALL
		SELECT distinct ne.AgreementKey, CASE WHEN c.AgreementKey IS NOT NULL THEN 1 ELSE 0 end as PrimaryContract_ktsx
		FROM NomikesAnagContractCMS ne
		LEFT JOIN NomikesAnagContractCMS c on c.AgreementKey = ne.AgreementKey AND c.ActivityDate = ne.ActivityDate AND c.ActivityTypeKey = ne.ActivityTypeKey AND ne.LegalActivityMapping = c.LegalActivityMapping
		WHERE ne.LegalActivityMapping = 'Κατάσχεση' 
  '''
)
PrimaryContractKTSX.createOrReplaceTempView("PrimaryContractKTSX")


##### 1754HistoricalAgg

In [0]:
Historical1754Agg = spark.sql(
f'''
		SELECT AgreementKey, max(ActivityDate) ActivityDate, ActivityMonth	
		FROM (	
		SELECT n.*, last_day(n.ActivityDate) ActivityMonth	
		FROM Nomikes n	
		LEFT JOIN NomikesAnagAFM c on c.ParentPartyId = n.ParentPartyId	
		WHERE n.ActivityTypeKey = '1754'	
		AND n.ActivityDate > '2021-12-31'	
		AND c.LegalActivityMapping = n.LegalActivityMapping	
		AND last_day(c.ActivityDate) < last_day(n.ActivityDate)
		--AND CMSFlag = 0
		) a	
		GROUP BY AgreementKey, ActivityMonth	
  '''
)
Historical1754Agg.createOrReplaceTempView("Historical1754Agg")

##### 1754Historical

In [0]:
Historical1754 = spark.sql(
f'''
		SELECT acc.ExposureKey,b.* 
		FROM (
		SELECT ParentPartyId, max(ActivityDate) ActivityDate, ActivityMonth
		FROM (
		SELECT n.*, last_day(n.ActivityDate) ActivityMonth
		FROM Nomikes n
		LEFT JOIN NomikesAnagAFM c on c.ParentPartyId = n.ParentPartyId
		WHERE n.ActivityTypeKey = '1754'
		AND n.ActivityDate > '2021-12-31'
		AND c.LegalActivityMapping = n.LegalActivityMapping
		AND last_day(c.ActivityDate) < last_day(n.ActivityDate)) a	
		GROUP BY ParentPartyId, ActivityMonth) b
		JOIN mastertable acc on acc.ParentPartyId = b.ParentPartyId
'''
)
Historical1754.createOrReplaceTempView("Historical1754")


##### NomikesPerimeterContract

In [0]:
NomikesPerimeterContract = spark.sql(
    '''
    SELECT
        AgreementKey,
        ExposureKey,
        CMSFlag,
        MIN(CASE WHEN LegalChapter = 'ΚΑΤΑ_ΔΠ' THEN ActivityDate END) AS `ΚΑΤΑ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΕΚΔ_ΔΠ' THEN ActivityDate END) AS `ΕΚΔ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΕΠΙ_ΔΠ' THEN ActivityDate END) AS `ΕΠΙ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΚΑΤΑΣΧ' THEN ActivityDate END) AS `ΚΑΤΑΣΧ`,
        MIN(CASE WHEN LegalChapter = 'ΚΑΤΑ_ΑΓΩΓ' THEN ActivityDate END) AS `ΚΑΤΑ_ΑΓΩΓ`,
        MIN(CASE WHEN LegalChapter = 'ΕΚΔ_ΑΓΩΓ' THEN ActivityDate END) AS `ΕΚΔ_ΑΓΩΓ`,
        MIN(CASE WHEN LegalChapter = 'ΚΟΙΝ_ΑΓΩΓ' THEN ActivityDate END) AS `ΚΟΙΝ_ΑΓΩΓ`,
        MIN(CASE WHEN LegalChapter = 'ΠΡΟΣΗΜ' THEN ActivityDate END) AS `ΠΡΟΣΗΜ`,
        MIN(CASE WHEN LegalChapter = 'ΚΑΤΑ_ΔΠ' THEN ExternalCompany END) AS `AG_ΚΑΤΑ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΕΚΔ_ΔΠ' THEN ExternalCompany END) AS `AG_ΕΚΔ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΕΠΙ_ΔΠ' THEN ExternalCompany END) AS `AG_ΕΠΙ_ΔΠ`,
        MIN(CASE WHEN LegalChapter = 'ΚΑΤΑΣΧ' THEN ExternalCompany END) AS `AG_ΚΑΤΑΣΧ`
    FROM NomikesAnag
    GROUP BY AgreementKey, ExposureKey, CMSFlag

    '''
)
NomikesPerimeterContract.createOrReplaceTempView("NomikesPerimeterContract")

##### LegalProcess 

In [0]:
LegalProcess = spark.sql(
f'''
select 
AgreementKey,
EkkrPairA_EntryDate_dp,
EkkrPairA_ExitDate_dp,
EkkrPairD_EntryDate_dp,
EkkrPairD_ExitDate_dp,
EkkrPairG_EntryDate_dp,
EkkrPairG_ExitDate_dp,
EkkrPairB_EntryDate_agog,
EkkrPairB_ExitDate_agog,
EkkrPairC_EntryDate_agog,
EkkrPairC_ExitDate_agog,
EkkrPairE_EntryDate_agog,
EkkrPairE_ExitDate_agog,
EkkrPairF_EntryDate_agog,
EkkrPairF_ExitDate_agog,
EkkrPairH_EntryDate_agog,
EkkrPairH_ExitDate_agog,
EkkrPairI_EntryDate_agog,
EkkrPairI_ExitDate_agog,
EkkrPairL_EntryDate_agog,
EkkrPairL_ExitDate_agog,
EkkrPairM_EntryDate_agog,
EkkrPairM_ExitDate_agog,
EkkrPairA_EntryDate_ktsx,
EkkrPairA_ExitDate_ktsx,
EkkrPairD_EntryDate_ktsx,
EkkrPairD_ExitDate_ktsx,
EkkrPairG_EntryDate_ktsx,
EkkrPairG_ExitDate_ktsx,
EkkrPairJ_EntryDate_ktsx,
EkkrPairJ_ExitDate_ktsx,
EkkrPairK_EntryDate_ktsx,
EkkrPairK_ExitDate_ktsx,
EkkrPairL_EntryDate_ktsx,
EkkrPairL_ExitDate_ktsx,
EkkrPairN_EntryDate_ktsx,
EkkrPairN_ExitDate_ktsx,
EkkrPairA_EntryDate_pros,
EkkrPairA_ExitDate_pros,
EkkrPairD_EntryDate_pros,
EkkrPairD_ExitDate_pros,
EkkrPairG_EntryDate_pros,
EkkrPairG_ExitDate_pros,
EkkrPairJ_EntryDate_pros,
EkkrPairJ_ExitDate_pros,
EkkrPairK_EntryDate_pros,
EkkrPairK_ExitDate_pros,
AssignmentDate_dp,
AssignmentDate_agog,
AssignmentDate_ktsx,
AssignmentDate_pros,
CurrentAgency_dp,
CurrentAgency_agog,
CurrentAgency_ktsx,
CurrentAgency_pros
from  `{env_prefix}-cdh-delivery`.repoint.core_legalmodule where BusinessDate = {run_date!r}
'''
)
LegalProcess.createOrReplaceTempView("LegalProcess")

##### EkkremotitesTimePeriods

In [0]:
EkkremotitesTimePeriods = spark.sql(
f'''
		SELECT * 
		FROM (
		SELECT AgreementKey, 'ΔΠ' LegalActivity, EkkrPairA_EntryDate_dp startDate, EkkrPairA_ExitDate_dp endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΔΠ' LegalActivity, EkkrPairD_EntryDate_dp startDate, EkkrPairD_ExitDate_dp endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΔΠ' LegalActivity, EkkrPairG_EntryDate_dp startDate, EkkrPairG_ExitDate_dp endDate
		FROM LegalProcess
		
		UNION ALL
		
		SELECT AgreementKey, 'ΑΓΩΓ' LegalActivity, EkkrPairB_EntryDate_agog startDate, EkkrPairB_ExitDate_agog endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΑΓΩΓ' LegalActivity, EkkrPairC_EntryDate_agog startDate, EkkrPairC_ExitDate_agog endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΑΓΩΓ' LegalActivity, EkkrPairE_EntryDate_agog startDate, EkkrPairE_ExitDate_agog endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΑΓΩΓ' LegalActivity, EkkrPairF_EntryDate_agog startDate, EkkrPairF_ExitDate_agog endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΑΓΩΓ' LegalActivity, EkkrPairH_EntryDate_agog startDate, EkkrPairH_ExitDate_agog endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΑΓΩΓ' LegalActivity, EkkrPairI_EntryDate_agog startDate, EkkrPairI_ExitDate_agog endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΑΓΩΓ' LegalActivity, EkkrPairL_EntryDate_agog startDate, EkkrPairL_ExitDate_agog endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΑΓΩΓ' LegalActivity, EkkrPairM_EntryDate_agog startDate, EkkrPairM_ExitDate_agog endDate
		FROM LegalProcess
		
		
		UNION ALL
		
		SELECT AgreementKey, 'ΚΑΤΑΣΧ' LegalActivity, EkkrPairA_EntryDate_ktsx startDate, EkkrPairA_ExitDate_ktsx endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΚΑΤΑΣΧ' LegalActivity, EkkrPairD_EntryDate_ktsx startDate, EkkrPairD_ExitDate_ktsx endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΚΑΤΑΣΧ' LegalActivity, EkkrPairG_EntryDate_ktsx startDate, EkkrPairG_ExitDate_ktsx endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΚΑΤΑΣΧ' LegalActivity, EkkrPairJ_EntryDate_ktsx startDate, EkkrPairJ_ExitDate_ktsx endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΚΑΤΑΣΧ' LegalActivity, EkkrPairK_EntryDate_ktsx startDate, EkkrPairK_ExitDate_ktsx endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΚΑΤΑΣΧ' LegalActivity, EkkrPairL_EntryDate_ktsx startDate, EkkrPairL_ExitDate_ktsx endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΚΑΤΑΣΧ' LegalActivity, EkkrPairN_EntryDate_ktsx startDate, EkkrPairN_ExitDate_ktsx endDate
		FROM LegalProcess
		
		
		UNION ALL
		
		SELECT AgreementKey, 'ΠΡΟΣ' LegalActivity, EkkrPairA_EntryDate_pros startDate, EkkrPairA_ExitDate_pros endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΠΡΟΣ' LegalActivity, EkkrPairD_EntryDate_pros startDate, EkkrPairD_ExitDate_pros endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΠΡΟΣ' LegalActivity, EkkrPairG_EntryDate_pros startDate, EkkrPairG_ExitDate_pros endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΠΡΟΣ' LegalActivity, EkkrPairJ_EntryDate_pros startDate, EkkrPairJ_ExitDate_pros endDate
		FROM LegalProcess
		UNION ALL
		SELECT AgreementKey, 'ΠΡΟΣ' LegalActivity, EkkrPairK_EntryDate_pros startDate, EkkrPairK_ExitDate_pros endDate
		FROM LegalProcess
		) a WHERE a.startDate IS NOT NULL
  '''
)
EkkremotitesTimePeriods.createOrReplaceTempView("EkkremotitesTimePeriods")

In [0]:
EkkremotitesTimePeriods = spark.sql(
    f'''
select AgreementKey, LegalActivity,  
StartDate, 
endDate 
from EkkremotitesTimePeriods

'''
)
EkkremotitesTimePeriods.createOrReplaceTempView("EkkremotitesTimePeriods")

##### TimePeriods

In [0]:
TimePeriods = spark.sql(
f'''
SELECT
AgreementKey, LegalActivity, startDate, endDate,
ROW_NUMBER() over (partition by AgreementKey, LegalActivity  order by startDate, endDate) as rn
FROM EkkremotitesTimePeriods
'''
)
TimePeriods.createOrReplaceTempView("TimePeriods")

In [0]:
depth = 10
EkkremotitesMergedTimePeriods_0 = recursive_union("TimePeriods", depth, 'TimePeriods')
EkkremotitesMergedTimePeriods_0.createOrReplaceTempView("EkkremotitesMergedTimePeriods_0")
EkkremotitesMergedTimePeriods = spark.sql(
f'''
SELECT
		AgreementKey, LegalActivity, GroupId, min(startDate) startDate, max(endDate) endDate
  FROM EkkremotitesMergedTimePeriods_0
		GROUP BY AgreementKey, LegalActivity, GroupId
		ORDER BY AgreementKey, LegalActivity, GroupId
'''
)
EkkremotitesMergedTimePeriods.createOrReplaceTempView("EkkremotitesMergedTimePeriods")
spark.sql(f"DROP TABLE IF EXISTS `{env_prefix}-cdh-delivery`.repoint.int_ra_legalActions_EkkremotitesMergedTimePeriods")
create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_ra_legalActions_EkkremotitesMergedTimePeriods", EkkremotitesMergedTimePeriods)
repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_ra_legalActions_EkkremotitesMergedTimePeriods")

##### EkkremotitesMergedTimePeriodsFinal

In [0]:
EkkremotitesMergedTimePeriodsFinal = spark.sql(
    f'''
SELECT b.AgreementKey
        ,ExposureKey
        ,`ΚΑΤΑ_ΔΠ_EkkrTime`
        ,`ΕΚΔ_ΔΠ_EkkrTime`
        ,`ΕΠΙ_ΔΠ_EkkrTime`
        ,`ΔΠ_ΑΝ_EkkrTime`
        
        ,`ΚΑΤΑΣΧ_EkkrTime`
        ,`ΚΑΤΑΣΧ_ΑΝ_EkkrTime`
        
        ,`ΚΑΤΑ_ΑΓΩΓ_EkkrTime`
        ,`ΕΚΔ_ΑΓΩΓ_EkkrTime`
        ,`ΚΟΙΝ_ΑΓΩΓ_EkkrTime`
        ,`ΑΓΩΓ_ΑΝ_EkkrTime`
        
        ,`ΠΡΟΣΗΜ_EkkrTime`
        ,`ΠΡΟΣΗΜ_ΑΝ_EkkrTime`
    FROM (
        SELECT AgreementKey
        ,sum(`ΚΑΤΑ_ΔΠ_EkkrTime`) `ΚΑΤΑ_ΔΠ_EkkrTime`
        ,sum(`ΕΚΔ_ΔΠ_EkkrTime`) `ΕΚΔ_ΔΠ_EkkrTime`
        ,sum(`ΕΠΙ_ΔΠ_EkkrTime`) `ΕΠΙ_ΔΠ_EkkrTime`
        ,sum(`ΔΠ_ΑΝ_EkkrTime`) `ΔΠ_ΑΝ_EkkrTime`
        
        ,sum(`ΚΑΤΑΣΧ_EkkrTime`) `ΚΑΤΑΣΧ_EkkrTime`
        ,sum(`ΚΑΤΑΣΧ_ΑΝ_EkkrTime`) `ΚΑΤΑΣΧ_ΑΝ_EkkrTime`
        
        ,sum(`ΚΑΤΑ_ΑΓΩΓ_EkkrTime`) `ΚΑΤΑ_ΑΓΩΓ_EkkrTime`
        ,sum(`ΕΚΔ_ΑΓΩΓ_EkkrTime`) `ΕΚΔ_ΑΓΩΓ_EkkrTime`
        ,sum(`ΚΟΙΝ_ΑΓΩΓ_EkkrTime`) `ΚΟΙΝ_ΑΓΩΓ_EkkrTime`
        ,sum(`ΑΓΩΓ_ΑΝ_EkkrTime`) `ΑΓΩΓ_ΑΝ_EkkrTime`
        
        ,sum(`ΠΡΟΣΗΜ_EkkrTime`) `ΠΡΟΣΗΜ_EkkrTime`
        ,sum(`ΠΡΟΣΗΜ_ΑΝ_EkkrTime`) `ΠΡΟΣΗΜ_ΑΝ_EkkrTime`
        FROM(
            SELECT t.AgreementKey
            ,CASE WHEN LegalActivity = 'ΔΠ' AND `ΚΑΤΑ_ΔΠ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΚΑΤΑ_ΔΠ`,'9999-12-31')>endDate, endDate, `ΚΑΤΑ_ΔΠ`)) end `ΚΑΤΑ_ΔΠ_EkkrTime`
            ,CASE WHEN LegalActivity = 'ΔΠ' AND `ΕΚΔ_ΔΠ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΕΚΔ_ΔΠ`,'9999-12-31')>endDate, endDate, `ΕΚΔ_ΔΠ`)) end `ΕΚΔ_ΔΠ_EkkrTime`
            ,CASE WHEN LegalActivity = 'ΔΠ' AND `ΕΠΙ_ΔΠ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΕΠΙ_ΔΠ`,'9999-12-31')>endDate, endDate, `ΕΠΙ_ΔΠ`)) end `ΕΠΙ_ΔΠ_EkkrTime`
            ,CASE WHEN LegalActivity = 'ΔΠ' AND `ΚΑΤΑ_ΔΠ` is NULL AND `ΕΚΔ_ΔΠ` is NULL AND `ΕΠΙ_ΔΠ` is NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate, endDate) end `ΔΠ_ΑΝ_EkkrTime`
            
            ,CASE WHEN LegalActivity = 'ΚΑΤΑΣΧ' AND `ΚΑΤΑΣΧ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΚΑΤΑΣΧ`,'9999-12-31')>endDate, endDate, `ΚΑΤΑΣΧ`)) end `ΚΑΤΑΣΧ_EkkrTime`
            ,CASE WHEN LegalActivity = 'ΚΑΤΑΣΧ' AND `ΚΑΤΑΣΧ` is NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate, endDate) end `ΚΑΤΑΣΧ_ΑΝ_EkkrTime`
            
            ,CASE WHEN LegalActivity = 'ΑΓΩΓ' AND `ΚΑΤΑ_ΑΓΩΓ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΚΑΤΑ_ΑΓΩΓ`,'9999-12-31')>endDate, endDate, `ΚΑΤΑ_ΑΓΩΓ`)) end `ΚΑΤΑ_ΑΓΩΓ_EkkrTime`
            ,CASE WHEN LegalActivity = 'ΑΓΩΓ' AND `ΕΚΔ_ΑΓΩΓ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΕΚΔ_ΑΓΩΓ`,'9999-12-31')>endDate, endDate, `ΕΚΔ_ΑΓΩΓ`)) end `ΕΚΔ_ΑΓΩΓ_EkkrTime`
            ,CASE WHEN LegalActivity = 'ΑΓΩΓ' AND `ΚΟΙΝ_ΑΓΩΓ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΚΟΙΝ_ΑΓΩΓ`,'9999-12-31')>endDate, endDate, `ΚΟΙΝ_ΑΓΩΓ`)) end `ΚΟΙΝ_ΑΓΩΓ_EkkrTime`
            ,CASE WHEN LegalActivity = 'ΑΓΩΓ' AND `ΚΑΤΑ_ΑΓΩΓ` is NULL AND `ΕΚΔ_ΑΓΩΓ` is NULL AND `ΚΟΙΝ_ΑΓΩΓ` is NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate, endDate) end `ΑΓΩΓ_ΑΝ_EkkrTime`
            
            ,CASE WHEN LegalActivity = 'ΠΡΟΣΗΜ' AND `ΠΡΟΣΗΜ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΠΡΟΣΗΜ`,'9999-12-31')>endDate, endDate, `ΠΡΟΣΗΜ`)) end `ΠΡΟΣΗΜ_EkkrTime`
            ,CASE WHEN LegalActivity = 'ΠΡΟΣΗΜ' AND `ΚΑΤΑΣΧ` is NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate, endDate) end `ΠΡΟΣΗΜ_ΑΝ_EkkrTime`
            
            FROM `{env_prefix}-cdh-delivery`.repoint.int_ra_legalActions_EkkremotitesMergedTimePeriods t
            JOIN (
                SELECT distinct AgreementKey FROM mastertable WHERE CMSFlag = 0
            ) acc on t.AgreementKey = acc.AgreementKey
            LEFT JOIN (SELECT distinct AgreementKey,`ΚΑΤΑ_ΔΠ`,`ΕΚΔ_ΔΠ`,`ΕΠΙ_ΔΠ`,`ΚΑΤΑΣΧ`,`ΚΑΤΑ_ΑΓΩΓ`,`ΕΚΔ_ΑΓΩΓ`,`ΚΟΙΝ_ΑΓΩΓ`,`ΠΡΟΣΗΜ`
                        FROM NomikesPerimeterContract 
                            ) n on n.AgreementKey = t.AgreementKey
            WHERE endDate IS NOT NULL
        ) a
        GROUP BY AgreementKey
        ) b
        JOIN mastertable acc on b.AgreementKey = acc.AgreementKey
        UNION ALL
        SELECT AgreementKey
        ,ExposureKey
        ,sum(`ΚΑΤΑ_ΔΠ_EkkrTime`) `ΚΑΤΑ_ΔΠ_EkkrTime`
        ,sum(`ΕΚΔ_ΔΠ_EkkrTime`) `ΕΚΔ_ΔΠ_EkkrTime`
        ,sum(`ΕΠΙ_ΔΠ_EkkrTime`) `ΕΠΙ_ΔΠ_EkkrTime`
        ,sum(`ΔΠ_ΑΝ_EkkrTime`) `ΔΠ_ΑΝ_EkkrTime`
        
        ,sum(`ΚΑΤΑΣΧ_EkkrTime`) `ΚΑΤΑΣΧ_EkkrTime`
        ,sum(`ΚΑΤΑΣΧ_ΑΝ_EkkrTime`) `ΚΑΤΑΣΧ_ΑΝ_EkkrTime`
        
        ,sum(`ΚΑΤΑ_ΑΓΩΓ_EkkrTime`) `ΚΑΤΑ_ΑΓΩΓ_EkkrTime`
        ,sum(`ΕΚΔ_ΑΓΩΓ_EkkrTime`) `ΕΚΔ_ΑΓΩΓ_EkkrTime`
        ,sum(`ΚΟΙΝ_ΑΓΩΓ_EkkrTime`) `ΚΟΙΝ_ΑΓΩΓ_EkkrTime`
        ,sum(`ΑΓΩΓ_ΑΝ_EkkrTime`) `ΑΓΩΓ_ΑΝ_EkkrTime`
        
        ,sum(`ΠΡΟΣΗΜ_EkkrTime`) `ΠΡΟΣΗΜ_EkkrTime`
        ,sum(`ΠΡΟΣΗΜ_ΑΝ_EkkrTime`) `ΠΡΟΣΗΜ_ΑΝ_EkkrTime`
        FROM(
        SELECT t.AgreementKey
        ,acc.ExposureKey
        ,CASE WHEN LegalActivity = 'ΔΠ' AND `ΚΑΤΑ_ΔΠ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΚΑΤΑ_ΔΠ`,'9999-12-31')>endDate, endDate, `ΚΑΤΑ_ΔΠ`)) end `ΚΑΤΑ_ΔΠ_EkkrTime`
        ,CASE WHEN LegalActivity = 'ΔΠ' AND `ΕΚΔ_ΔΠ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΕΚΔ_ΔΠ`,'9999-12-31')>endDate, endDate, `ΕΚΔ_ΔΠ`)) end `ΕΚΔ_ΔΠ_EkkrTime`
        ,CASE WHEN LegalActivity = 'ΔΠ' AND `ΕΠΙ_ΔΠ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΕΠΙ_ΔΠ`,'9999-12-31')>endDate, endDate, `ΕΠΙ_ΔΠ`)) end `ΕΠΙ_ΔΠ_EkkrTime`
        ,CASE WHEN LegalActivity = 'ΔΠ' AND `ΚΑΤΑ_ΔΠ` is NULL AND `ΕΚΔ_ΔΠ` is NULL AND `ΕΠΙ_ΔΠ` is NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate, endDate) end `ΔΠ_ΑΝ_EkkrTime`
        
        ,CASE WHEN LegalActivity = 'ΚΑΤΑΣΧ' AND `ΚΑΤΑΣΧ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΚΑΤΑΣΧ`,'9999-12-31')>endDate, endDate, `ΚΑΤΑΣΧ`)) end `ΚΑΤΑΣΧ_EkkrTime`
        ,CASE WHEN LegalActivity = 'ΚΑΤΑΣΧ' AND `ΚΑΤΑΣΧ` is NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate, endDate) end `ΚΑΤΑΣΧ_ΑΝ_EkkrTime`
        
        ,CASE WHEN LegalActivity = 'ΑΓΩΓ' AND `ΚΑΤΑ_ΑΓΩΓ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΚΑΤΑ_ΑΓΩΓ`,'9999-12-31')>endDate, endDate, `ΚΑΤΑ_ΑΓΩΓ`)) end `ΚΑΤΑ_ΑΓΩΓ_EkkrTime`
        ,CASE WHEN LegalActivity = 'ΑΓΩΓ' AND `ΕΚΔ_ΑΓΩΓ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΕΚΔ_ΑΓΩΓ`,'9999-12-31')>endDate, endDate, `ΕΚΔ_ΑΓΩΓ`)) end `ΕΚΔ_ΑΓΩΓ_EkkrTime`
        ,CASE WHEN LegalActivity = 'ΑΓΩΓ' AND `ΚΟΙΝ_ΑΓΩΓ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΚΟΙΝ_ΑΓΩΓ`,'9999-12-31')>endDate, endDate, `ΚΟΙΝ_ΑΓΩΓ`)) end `ΚΟΙΝ_ΑΓΩΓ_EkkrTime`
        ,CASE WHEN LegalActivity = 'ΑΓΩΓ' AND `ΚΑΤΑ_ΑΓΩΓ` is NULL AND `ΕΚΔ_ΑΓΩΓ` is NULL AND `ΚΟΙΝ_ΑΓΩΓ` is NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate, endDate) end `ΑΓΩΓ_ΑΝ_EkkrTime`
        
        ,CASE WHEN LegalActivity = 'ΠΡΟΣΗΜ' AND `ΠΡΟΣΗΜ` IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate,if(ifnull(`ΠΡΟΣΗΜ`,'9999-12-31')>endDate, endDate, `ΠΡΟΣΗΜ`)) end `ΠΡΟΣΗΜ_EkkrTime`
        ,CASE WHEN LegalActivity = 'ΠΡΟΣΗΜ' AND `ΚΑΤΑΣΧ` is NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',startDate, endDate) end `ΠΡΟΣΗΜ_ΑΝ_EkkrTime`
        
        FROM `{env_prefix}-cdh-delivery`.repoint.int_ra_legalActions_EkkremotitesMergedTimePeriods t
        JOIN mastertable acc on t.AgreementKey = acc.AgreementKey
        LEFT JOIN (SELECT distinct ExposureKey,`ΚΑΤΑ_ΔΠ`,`ΕΚΔ_ΔΠ`,`ΕΠΙ_ΔΠ`,`ΚΑΤΑΣΧ`,`ΚΑΤΑ_ΑΓΩΓ`,`ΕΚΔ_ΑΓΩΓ`,`ΚΟΙΝ_ΑΓΩΓ`,`ΠΡΟΣΗΜ` 
                     FROM NomikesPerimeterContract  ) n on n.ExposureKey = acc.ExposureKey
        WHERE endDate IS NOT NULL
        AND CMSFlag = 1) a
        GROUP BY AgreementKey, ExposureKey
    '''
)
EkkremotitesMergedTimePeriodsFinal.createOrReplaceTempView("EkkremotitesMergedTimePeriodsFinal")

##### NomikesFinal

In [0]:
NomikesFinal = spark.sql(
f'''
SELECT
        {run_date!r} as BusinessDate
        ,acc.CDI
        ,acc.PartyTaxRegistrationNumber
        ,acc.ParentPartyId
        ,ep.AgreementKey
        ,acc.ExposureKey
        ,acc.GeminiAccFlag
        ,acc.AccountStatus ArrearsStatusENG
        ,acc.AssetSegment
        ,acc.Balance
        ,acc.OnBalanceAmount
        ,acc.OffBalanceAmount         
        , l.agency  as Agency 
        ,lp.AssignmentDate_dp
        ,lp.AssignmentDate_agog
        ,lp.AssignmentDate_ktsx
        ,lp.AssignmentDate_pros
        ,lp.CurrentAgency_dp
        ,lp.CurrentAgency_agog
        ,lp.CurrentAgency_ktsx
        ,lp.CurrentAgency_pros
        ,`ΚΑΤΑ_ΔΠ`
        ,`ΕΚΔ_ΔΠ`
        ,`ΕΠΙ_ΔΠ`
        ,`ΚΑΤΑΣΧ`
        ,`ΚΑΤΑ_ΑΓΩΓ`
        ,`ΕΚΔ_ΑΓΩΓ`
        ,`ΚΟΙΝ_ΑΓΩΓ`
        ,`ΠΡΟΣΗΜ`
        ,`AG_ΚΑΤΑ_ΔΠ`
        ,`AG_ΕΚΔ_ΔΠ`
        ,`AG_ΕΠΙ_ΔΠ`
        ,`AG_ΚΑΤΑΣΧ`
        ,`PrimaryContract_ktsx`
        
        ,CASE WHEN `ΚΑΤΑ_ΔΠ` IS NOT NULL AND lp.AssignmentDate_dp IS NOT NULL AND lp.AssignmentDate_dp <= `ΚΑΤΑ_ΔΠ` THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_dp,`ΚΑΤΑ_ΔΠ`) -ifnull(`ΚΑΤΑ_ΔΠ_EkkrTime`,0)  end `ΚΑΤΑ_ΔΠ_Time`
        ,CASE WHEN `ΕΚΔ_ΔΠ` IS NOT NULL AND lp.AssignmentDate_dp IS NOT NULL AND lp.AssignmentDate_dp <= `ΕΚΔ_ΔΠ` THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_dp,`ΕΚΔ_ΔΠ`) -ifnull(`ΕΚΔ_ΔΠ_EkkrTime`,0)  end `ΕΚΔ_ΔΠ_Time`
        ,CASE WHEN `ΕΠΙ_ΔΠ` IS NOT NULL AND lp.AssignmentDate_dp IS NOT NULL AND lp.AssignmentDate_dp <= `ΕΠΙ_ΔΠ` THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_dp,`ΕΠΙ_ΔΠ`) -ifnull(`ΕΠΙ_ΔΠ_EkkrTime`,0)  end `ΕΠΙ_ΔΠ_Time`
        ,CASE WHEN COALESCE(`ΚΑΤΑ_ΔΠ`,`ΕΚΔ_ΔΠ`,`ΕΠΙ_ΔΠ`,'1900-01-01') = '1900-01-01' AND lp.AssignmentDate_dp IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_dp,{run_date!r}) -ifnull(`ΔΠ_ΑΝ_EkkrTime`,0)  end `Ekkr_ΔΠ_Time`
        
        ,CASE WHEN `ΚΑΤΑ_ΑΓΩΓ` IS NOT NULL AND lp.AssignmentDate_agog IS NOT NULL AND lp.AssignmentDate_agog <= `ΚΑΤΑ_ΑΓΩΓ` THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_agog,`ΚΑΤΑ_ΑΓΩΓ`) -ifnull(`ΚΑΤΑ_ΑΓΩΓ_EkkrTime`,0) end `ΚΑΤΑ_ΑΓΩΓ_Time`
        ,CASE WHEN `ΕΚΔ_ΑΓΩΓ` IS NOT NULL AND lp.AssignmentDate_agog IS NOT NULL AND lp.AssignmentDate_agog <= `ΕΚΔ_ΑΓΩΓ` THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_agog,`ΕΚΔ_ΑΓΩΓ`) -ifnull(`ΕΚΔ_ΑΓΩΓ_EkkrTime`,0) end `ΕΚΔ_ΑΓΩΓ_Time`
        ,CASE WHEN `ΚΟΙΝ_ΑΓΩΓ` IS NOT NULL AND lp.AssignmentDate_agog IS NOT NULL AND lp.AssignmentDate_agog <= `ΚΟΙΝ_ΑΓΩΓ` THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_agog,`ΚΟΙΝ_ΑΓΩΓ`) -ifnull(`ΚΟΙΝ_ΑΓΩΓ_EkkrTime`,0) end `ΚΟΙΝ_ΑΓΩΓ_Time`
        ,CASE WHEN COALESCE(`ΚΑΤΑ_ΑΓΩΓ`,`ΕΚΔ_ΑΓΩΓ`,`ΚΟΙΝ_ΑΓΩΓ`,'1900-01-01') = '1900-01-01' AND lp.AssignmentDate_agog IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_agog,{run_date!r}) -ifnull(`ΑΓΩΓ_ΑΝ_EkkrTime`,0) end `Ekkr_ΠΑΓ_Time`
        
        ,CASE WHEN `ΚΑΤΑΣΧ` IS NOT NULL AND lp.AssignmentDate_ktsx IS NOT NULL AND lp.AssignmentDate_ktsx <= `ΚΑΤΑΣΧ` THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_ktsx,`ΚΑΤΑΣΧ`) -ifnull(`ΚΑΤΑΣΧ_EkkrTime`,0) end `ΚΑΤΑΣΧ_Time`
        ,CASE WHEN `ΚΑΤΑΣΧ` is NULL AND lp.AssignmentDate_ktsx IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_ktsx,{run_date!r}) -ifnull(`ΚΑΤΑΣΧ_ΑΝ_EkkrTime`,0) end `Ekkr_KΣΧ_Time`
        
        ,CASE WHEN `ΠΡΟΣΗΜ` IS NOT NULL AND lp.AssignmentDate_pros IS NOT NULL AND lp.AssignmentDate_pros <= `ΠΡΟΣΗΜ` THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_pros,`ΠΡΟΣΗΜ`) -ifnull(`ΠΡΟΣΗΜ_EkkrTime`,0) end `ΠΡΟΣ_Time`
        ,CASE WHEN `ΠΡΟΣΗΜ` is NULL AND lp.AssignmentDate_pros IS NOT NULL THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_ktsx,{run_date!r}) -ifnull(`ΠΡΟΣΗΜ_ΑΝ_EkkrTime`,0) end `Ekkr_ΠΡΟΣ_Time`
        ,'Total Legal Actions' `PopulationType`
        FROM NomikesPerimeterContract ep 
        JOIN mastertable acc on ep.ExposureKey = acc.ExposureKey
        LEFT JOIN agent l on acc.ExposureKey = l.ExposureKey AND acc.CDI = l.CDI 
        -- LEFT JOIN NomikesAgency ng on ng.ExposureKey = ep.ExposureKey
        LEFT JOIN EkkremotitesMergedTimePeriodsFinal t on t.ExposureKey = ep.ExposureKey
        LEFT JOIN PrimaryContractktsx prm on prm.AgreementKey = ep.AgreementKey
        LEFT JOIN LegalProcess lp on lp.AgreementKey = ep.AgreementKey
        
        UNION ALL
        
        SELECT
        {run_date!r} as BusinessDate
        ,acc.CDI
        ,acc.PartyTaxRegistrationNumber
        ,acc.ParentPartyId
        ,acc.Agreementkey
        ,acc.ExposureKey
        ,acc.GeminiAccFlag
        ,acc.AccountStatus ArrearsStatusENG
        ,acc.AssetSegment
        ,acc.Balance
        ,acc.OnBalanceAmount
        ,acc.OffBalanceAmount
        , l.Agency as Agency
        ,NULL AssignmentDate_dp
        ,NULL AssignmentDate_agog
       ,lp.AssignmentDate_ktsx
        ,NULL AssignmentDate_pros
        ,NULL CurrentAgency_dp
        ,NULL CurrentAgency_agog
        ,lp.CurrentAgency_ktsx
        ,NULL CurrentAgency_pros
        ,NULL `ΚΑΤΑ_ΔΠ`
        ,NULL `ΕΚΔ_ΔΠ`
        ,NULL `ΕΠΙ_ΔΠ`
        ,ep.ActivityDate `ΚΑΤΑΣΧ`
        ,NULL `ΚΑΤΑ_ΑΓΩΓ`
        ,NULL `ΕΚΔ_ΑΓΩΓ`
        ,NULL `ΚΟΙΝ_ΑΓΩΓ`
        ,NULL `ΠΡΟΣΗΜ`
        ,NULL `AG_ΚΑΤΑ_ΔΠ`
        ,NULL `AG_ΕΚΔ_ΔΠ`
        ,NULL `AG_ΕΠΙ_ΔΠ`
        ,NULL `AG_ΚΑΤΑΣΧ`
        ,CASE WHEN agr.AgreementKey IS NOT NULL THEN 1 ELSE 0 end `PrimaryContract_ktsx`
        
        ,NULL `ΚΑΤΑ_ΔΠ_Time`
        ,NULL `ΕΚΔ_ΔΠ_Time`
        ,NULL `ΕΠΙ_ΔΠ_Time`
        ,NULL `Ekkr_ΔΠ_Time`
        
        ,NULL `ΚΑΤΑ_ΑΓΩΓ_Time`
        ,NULL `ΕΚΔ_ΑΓΩΓ_Time`
        ,NULL `ΚΟΙΝ_ΑΓΩΓ_Time`
        ,NULL `Ekkr_ΠΑΓ_Time`
        
        ,CASE WHEN ep.ActivityDate IS NOT NULL AND lp.AssignmentDate_ktsx IS NOT NULL AND lp.AssignmentDate_ktsx <= ep.ActivityDate THEN `{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_ktsx,ep.ActivityDate) -ifnull(`ΚΑΤΑΣΧ_EkkrTime`,0) end `ΚΑΤΑΣΧ_Time`
        ,NULL `Ekkr_KΣΧ_Time`
        
        ,NULL `ΠΡΟΣ_Time`
        ,NULL `Ekkr_ΠΡΟΣ_Time`
        ,'1754 Historical' `PopulationType`
        FROM Historical1754 ep 
        JOIN mastertable acc on ep.ExposureKey = acc.ExposureKey
        LEFT JOIN agent l on acc.ExposureKey = l.ExposureKey AND acc.CDI = l.CDI 
        LEFT JOIN EkkremotitesMergedTimePeriodsFinal t on t.ExposureKey = acc.ExposureKey
        LEFT JOIN LegalProcess lp on lp.AgreementKey = acc.AgreementKey
        LEFT JOIN Historical1754Agg agr on acc.AgreementKey = agr.AgreementKey AND agr.ActivityMonth = ep.ActivityMonth AND agr.ActivityDate = ep.ActivityDate
        
        UNION ALL 
        
        SELECT
        {run_date!r} as BusinessDate
        ,NULL CDI
        ,NULL PartyTaxRegistrationNumber
        ,NULL ParentPartyId
        ,lp.AgreementKey AgreementKey
        ,acc.ExposureKey
        ,acc.GeminiAccFlag
        ,NULL ArrearsStatusENG
        ,NULL AssetSegment
        ,NULL Balance
        ,NULL OnBalanceAmount
        ,NULL OffBalanceAmount           
        ,NULL AgencyCode 
        ,lp.AssignmentDate_dp
        ,lp.AssignmentDate_agog
        ,lp.AssignmentDate_ktsx
        ,lp.AssignmentDate_pros
        ,lp.CurrentAgency_dp
        ,lp.CurrentAgency_agog
        ,lp.CurrentAgency_ktsx
        ,lp.CurrentAgency_pros
        ,NULL `ΚΑΤΑ_ΔΠ`
        ,NULL `ΕΚΔ_ΔΠ`
        ,NULL `ΕΠΙ_ΔΠ`
        ,NULL `ΚΑΤΑΣΧ`
        ,NULL `ΚΑΤΑ_ΑΓΩΓ`
        ,NULL `ΕΚΔ_ΑΓΩΓ`
        ,NULL `ΚΟΙΝ_ΑΓΩΓ`
        ,NULL `ΠΡΟΣΗΜ`
        ,NULL `AG_ΚΑΤΑ_ΔΠ`
        ,NULL `AG_ΕΚΔ_ΔΠ`
        ,NULL `AG_ΕΠΙ_ΔΠ`
        ,NULL `AG_ΚΑΤΑΣΧ`
        ,NULL `PrimaryContract_ktsx`
        
        ,NULL `ΚΑΤΑ_ΔΠ_Time`
        ,NULL `ΕΚΔ_ΔΠ_Time`
        ,NULL `ΕΠΙ_ΔΠ_Time`
        ,`{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_dp,{run_date!r}) -ifnull(`ΔΠ_ΑΝ_EkkrTime`,0) `Ekkr_ΔΠ_Time`
        
        ,NULL `ΚΑΤΑ_ΑΓΩΓ_Time`
        ,NULL `ΕΚΔ_ΑΓΩΓ_Time`
        ,NULL `ΚΟΙΝ_ΑΓΩΓ_Time`
        ,`{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_agog,{run_date!r}) -ifnull(`ΑΓΩΓ_ΑΝ_EkkrTime`,0) `Ekkr_ΠΑΓ_Time`
        
        ,NULL `ΚΑΤΑΣΧ_Time`
        ,`{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_ktsx,{run_date!r}) -ifnull(`ΚΑΤΑΣΧ_ΑΝ_EkkrTime`,0) `Ekkr_KΣΧ_Time`
        
        ,NULL `ΠΡΟΣ_Time`
        ,`{env_prefix}-cdh-delivery`.repoint.GetTimeDurationInBusinessDays('dd',lp.AssignmentDate_ktsx,{run_date!r}) -ifnull(`ΠΡΟΣΗΜ_ΑΝ_EkkrTime`,0) `Ekkr_ΠΡΟΣ_Time`
        ,'Total Legal Process' `PopulationType`
        FROM LegalProcess lp
        JOIN mastertable acc on acc.AgreementKey = lp.AgreementKey
        LEFT JOIN EkkremotitesMergedTimePeriodsFinal t on t.Exposurekey = acc.Exposurekey
  '''
)
NomikesFinal.createOrReplaceTempView("NomikesFinal")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT DISTINCT 
    CAST(BusinessDate AS DATE) AS BusinessDate,
    CAST(nf.CDI AS STRING) AS CDI,
    CAST(nf.PartyTaxRegistrationNumber AS STRING) AS PartyTaxRegistrationNumber,
    CAST(nf.ParentPartyId AS STRING) AS ParentPartyId,
    CAST(nf.AgreementKey AS STRING) AS AgreementKey,
    CAST(nf.ExposureKey AS STRING) AS ExposureKey,
    CAST(ArrearsStatusENG AS STRING) AS ArrearsStatusENG,
    CAST(AssetSegment AS STRING) AS AssetSegment,
    CAST(Balance AS DECIMAL(19,2)) AS Balance,
    CAST(OnBalanceAmount AS DECIMAL(19,2)) AS OnBalanceAmount,
    CAST(OffBalanceAmount AS DECIMAL(19,2)) AS OffBalanceAmount,
    CAST(Agency AS STRING) AS AgencyCode,
    CAST(AssignmentDate_dp AS DATE) AS AssignmentDate_dp,
    CAST(AssignmentDate_agog AS DATE) AS AssignmentDate_agog,
    CAST(AssignmentDate_ktsx AS DATE) AS AssignmentDate_ktsx,
    CAST(AssignmentDate_pros AS DATE) AS AssignmentDate_pros,
    CAST(CurrentAgency_dp AS STRING) AS CurrentAgency_dp,
    CAST(CurrentAgency_agog AS STRING) AS CurrentAgency_agog,
    CAST(CurrentAgency_ktsx AS STRING) AS CurrentAgency_ktsx,
    CAST(CurrentAgency_pros AS STRING) AS CurrentAgency_pros,
    CAST(`ΚΑΤΑ_ΔΠ` AS DATE) AS `ΚΑΤΑ_ΔΠ`,
    CAST(`ΕΚΔ_ΔΠ` AS DATE) AS `ΕΚΔ_ΔΠ`,
    CAST(`ΕΠΙ_ΔΠ` AS DATE) AS `ΕΠΙ_ΔΠ`,
    CAST(`ΚΑΤΑΣΧ` AS DATE) AS `ΚΑΤΑΣΧ`,
    CAST(`ΚΑΤΑ_ΑΓΩΓ` AS DATE) AS `ΚΑΤΑ_ΑΓΩΓ`,
    CAST(`ΕΚΔ_ΑΓΩΓ` AS DATE) AS `ΕΚΔ_ΑΓΩΓ`,
    CAST(`ΚΟΙΝ_ΑΓΩΓ` AS DATE) AS `ΚΟΙΝ_ΑΓΩΓ`,
    CAST(`ΠΡΟΣΗΜ` AS DATE) AS `ΠΡΟΣΗΜ`,
    CAST(`AG_ΚΑΤΑ_ΔΠ` AS STRING) AS `AG_ΚΑΤΑ_ΔΠ`,
    CAST(`AG_ΕΚΔ_ΔΠ` AS STRING) AS `AG_ΕΚΔ_ΔΠ`,
    CAST(`AG_ΕΠΙ_ΔΠ` AS STRING) AS `AG_ΕΠΙ_ΔΠ`,
    CAST(`AG_ΚΑΤΑΣΧ` AS STRING) AS `AG_ΚΑΤΑΣΧ`,
    CAST(PrimaryContract_ktsx AS INT) AS PrimaryContract_ktsx,
    CAST(`ΚΑΤΑ_ΔΠ_Time` AS INT) AS `ΚΑΤΑ_ΔΠ_Time`,
    CAST(`ΕΚΔ_ΔΠ_Time` AS INT) AS `ΕΚΔ_ΔΠ_Time`,
    CAST(`ΕΠΙ_ΔΠ_Time` AS INT) AS `ΕΠΙ_ΔΠ_Time`,
    CAST(`Ekkr_ΔΠ_Time` AS INT) AS `Ekkr_ΔΠ_Time`,
    CAST(`ΚΑΤΑ_ΑΓΩΓ_Time` AS INT) AS `ΚΑΤΑ_ΑΓΩΓ_Time`,
    CAST(`ΕΚΔ_ΑΓΩΓ_Time` AS INT) AS `ΕΚΔ_ΑΓΩΓ_Time`,
    CAST(`ΚΟΙΝ_ΑΓΩΓ_Time` AS INT) AS `ΚΟΙΝ_ΑΓΩΓ_Time`,
    CAST(`Ekkr_ΠΑΓ_Time` AS INT) AS `Ekkr_ΠΑΓ_Time`,
    CAST(`ΚΑΤΑΣΧ_Time` AS INT) AS `ΚΑΤΑΣΧ_Time`,
    CAST(`Ekkr_KΣΧ_Time` AS INT) AS `Ekkr_KΣΧ_Time`,
    CAST(`ΠΡΟΣ_Time` AS INT) AS `ΠΡΟΣ_Time`,
    CAST(`Ekkr_ΠΡΟΣ_Time` AS INT) AS `Ekkr_ΠΡΟΣ_Time`,
    CAST(nf.PopulationType AS STRING) AS PopulationType,
    CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM NomikesFinal nf 
''')
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_LegalActions: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = []
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
LegalActions = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
LegalActions.createOrReplaceTempView("LegalActions")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/LegalActions/"
file_name = "LegalActions_" +run_date+ ".csv"
LegalActions.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)